# MHEMe Evaluation

## 0. Introduction

How well does our model perform? Let's discover it!

The goal of this notebook is to train several models for the several datasets, plots some results and try to understand if our model really stands out.

The models implemented will be characterized by:
1. General approach
    - MHEMe models
    - Direct models
2. Type of direct model
    - Temporary Convolutional Network (TCN)
    - XGBoost
    - ARIMA (only for baseline)
    - SARIMA (only for baseline)
3. [Type of training objective](Horizon%20Aware%20Loss.ipynb#Horizon-Aware-Huber-Loss)
    - Mean Squared Error (MSE)
    - Horizon Aware Huber Loss (HAH)
4. [Type of weighting strategy](Horizon%20Aware%20Loss.ipynb#Possible-weighting-strategies)
    - Uniform (U)
    - Soft Linear (SO)
    - Strong Linear (ST)
    - Exponential (E)

All the created models will be trained on a single time series from all the benchmark datasets presented in the [Exploratory Data Analysis notebook](eda.ipynb#Dataset-import-and-description), and will be tested using their respective loss as comparison metric.

See the quoted notebooks for more references.

The analysis will try to answer the following questions:
1. **Overall performance**: is using MHEMe actually *useful*?
2. **Variance of predictors**: are the single models learning something different?
3. **Relevance of weighting strategy**: does the choice affect the performance?
4. **Generalization**: are the dynamics learnt someway *general*?


## 1. Imports and Data Loading

In [ ]:
# Imports
import pandas as pd
import numpy as np
import os

from src.data_handler import *
from src.config_files import *
from src.direct_models import *
from src.mheme import *

Microsoft Visual C++ Redistributable is not installed, this may lead to the DLL load failure.
It can be downloaded at https://aka.ms/vs/16/release/vc_redist.x64.exe


OSError: [WinError 126] Impossibile trovare il modulo specificato. Error loading "c:\Users\enric\Documents\MHEMe\Multiple-Horizons-Ensemble-Method\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# Global variables
WINDOW = 48
HORIZON = 12

DATA_PATH = '../data'

TCN_PATH_CONFIG_LOAD = '../src/config_files/tcn_config.json'
TCN_PATH_SAVE = '../models/tcn'

XGB_PATH_CONFIG_LOAD = '../src/config_files/xgb_config.json'
XGB_PATH_SAVE = '../models/xgb'

In [ ]:
# Load data
dataset_initials = {'e': 'electricity', 's': 'solar', 't': 'traffic', 'v': 'volatility', 'w': 'wind'}

for ds in dataset_initials:
    print(f"Loading dataset: {dataset_initials[ds]}")
    X, data = data_loader(data_path = DATA_PATH, dataset = dataset_initials[ds])

    # Change name of X, data based on ds (This is pure flex):
    globals()[f'X_{ds}'] = X
    globals()[f'data_{ds}'] = data


Loading dataset: e
Loading dataset: s
Loading dataset: t
Loading dataset: v
Loading dataset: w


In [ ]:
# Preprocess data
for ds in dataset_initials:
    print(f"Preprocessing dataset: {dataset_initials[ds]}")
    X = globals()[f'X_{ds}']
    data = globals()[f'data_{ds}']
    
    X_slide, y_slide = sliding_window(X, window=WINDOW, horizon=HORIZON)
    train, val, test = train_validation_test_split(X_slide, y_slide)  

    # Change name of splits based on ds (This is pure flex):
    globals()[f'train_{ds}'] = train
    globals()[f'val_{ds}'] = val
    globals()[f'test_{ds}'] = test


Preprocessing dataset: electricity
Preprocessing dataset: solar
Preprocessing dataset: traffic
Preprocessing dataset: volatility
Preprocessing dataset: wind


## 2. Models training

### 2.1 Electricity Dataset

In [ ]:
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_tcn_hah_u_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_so_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_st_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_tcn_hah_s_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_xgb_hah_u_e  = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_so_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_st_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_xgb_hah_e_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_tcn_mse_u_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")
mheme_tcn_mse_so_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_tcn_mse_st_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")
mheme_tcn_mse_e_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_xgb_mse_u_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")
mheme_xgb_mse_so_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)  
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_xgb_mse_st_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")
mheme_xgb_mse_e_e = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
tcn_hah_e = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
tcn_mse_e = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
xgb_mse_e = XGBoost(horizon=HORIZON, file_path=XGB_PATH_CONFIG_LOAD)

models_e = {
    'mheme_tcn_hah_u_e': mheme_tcn_hah_u_e,
    'mheme_tcn_hah_so_e': mheme_tcn_hah_so_e,
    'mheme_tcn_hah_st_e': mheme_tcn_hah_st_e,
    'mheme_tcn_hah_s_e': mheme_tcn_hah_s_e,
    'mheme_xgb_hah_u_e': mheme_xgb_hah_u_e,
    'mheme_xgb_hah_so_e': mheme_xgb_hah_so_e,
    'mheme_xgb_hah_st_e': mheme_xgb_hah_st_e,
    'mheme_xgb_hah_e_e': mheme_xgb_hah_e_e,
    'mheme_tcn_mse_u_e': mheme_tcn_mse_u_e
    'mheme_tcn_mse_so_e': mheme_tcn_mse_so_e,
    'mheme_tcn_mse_st_e': mheme_tcn_mse_st_e,
    'mheme_tcn_mse_e_e': mheme_tcn_mse_e_e,
    'mheme_xgb_mse_u_e': mheme_xgb_mse_u_e,
    'mheme_xgb_mse_so_e': mheme_xgb_mse_so_e,
    'mheme_xgb_mse_st_e': mheme_xgb_mse_st_e,
    'mheme_xgb_mse_e_e': mheme_xgb_mse_e_e,
    'tcn_hah_e': tcn_hah_e,
    'tcn_mse_e': tcn_mse_e,
    'xgb_mse_e': xgb_mse_e
}            

In [ ]:
# Fit models
for name, model in models_e.items():
    print(f"Fitting model: {name}")
    train = globals()[f'train_e']
    model.fit(train[0], train[1])

In [ ]:
# Save models
for name, model in models_e.items():
    print(f"Saving model: {name}")
    model.save_model(f'../models/electricity/{name}')

### 2.2 Solar Dataset

In [ ]:
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_tcn_hah_u_s = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_so_s = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_st_s = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_tcn_hah_e_s = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_xgb_hah_u_s  = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_so_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_st_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_xgb_hah_e_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_tcn_mse_u_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")
mheme_tcn_mse_so_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_tcn_mse_st_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")
mheme_tcn_mse_e_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_xgb_mse_u_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")   
mheme_xgb_mse_so_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_xgb_mse_st_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")    
mheme_xgb_mse_e_s = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
tcn_hah_s = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
tcn_mse_s = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
xgb_mse_s = XGBoost(horizon=HORIZON, file_path=XGB_PATH_CONFIG_LOAD)

models_s = {
    'mheme_tcn_hah_u_s': mheme_tcn_hah_u_s,
    'mheme_tcn_hah_so_s': mheme_tcn_hah_so_s,
    'mheme_tcn_hah_st_s': mheme_tcn_hah_st_s,
    'mheme_tcn_hah_e_s': mheme_tcn_hah_e_s,
    'mheme_xgb_hah_u_s': mheme_xgb_hah_u_s,
    'mheme_xgb_hah_so_s': mheme_xgb_hah_so_s,
    'mheme_xgb_hah_st_s': mheme_xgb_hah_st_s,
    'mheme_xgb_hah_e_s': mheme_xgb_hah_e_s,
    'mheme_tcn_mse_u_s': mheme_tcn_mse_u_s,
    'mheme_tcn_mse_so_s': mheme_tcn_mse_so_s,
    'mheme_tcn_mse_st_s': mheme_tcn_mse_st_s,
    'mheme_tcn_mse_e_s': mheme_tcn_mse_e_s,
    'mheme_xgb_mse_u_s': mheme_xgb_mse_u_s,
    'mheme_xgb_mse_so_s': mheme_xgb_mse_so_s,
    'mheme_xgb_mse_st_s': mheme_xgb_mse_st_s,
    'mheme_xgb_mse_e_s': mheme_xgb_mse_e_s,
    'tcn_hah_s': tcn_hah_s,
    'tcn_mse_s': tcn_mse_s,
    'xgb_mse_s': xgb_mse_s
}

In [ ]:
# Fit models
for name, model in models_s.items():
    print(f"Fitting model: {name}")
    train = globals()[f'train_s']
    model.fit(train[0], train[1])

In [ ]:
# Save models
for name, model in models_s.items():
    print(f"Saving model: {name}")
    model.save_model(f'../models/solar/{name}.pkl')

### 2.3 Traffic Dataset

In [ ]:
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_tcn_hah_u_t = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_so_t = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_st_t = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_tcn_hah_e_t = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_xgb_hah_u_t  = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_so_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_st_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_xgb_hah_e_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_tcn_mse_u_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")
mheme_tcn_mse_so_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_tcn_mse_st_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")
mheme_tcn_mse_e_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_xgb_mse_u_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")   
mheme_xgb_mse_so_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_xgb_mse_st_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")    
mheme_xgb_mse_e_t = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
tcn_hah_t = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
tcn_mse_t = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
xgb_mse_t = XGBoost(horizon=HORIZON, file_path=XGB_PATH_CONFIG_LOAD)

models_t = {
    'mheme_tcn_hah_u_t': mheme_tcn_hah_u_t,
    'mheme_tcn_hah_so_t': mheme_tcn_hah_so_t,
    'mheme_tcn_hah_st_t': mheme_tcn_hah_st_t,
    'mheme_tcn_hah_e_t': mheme_tcn_hah_e_t,
    'mheme_xgb_hah_u_t': mheme_xgb_hah_u_t,
    'mheme_xgb_hah_so_t': mheme_xgb_hah_so_t,
    'mheme_xgb_hah_st_t': mheme_xgb_hah_st_t,
    'mheme_xgb_hah_e_t': mheme_xgb_hah_e_t,
    'mheme_tcn_mse_u_t': mheme_tcn_mse_u_t,
    'mheme_tcn_mse_to_t': mheme_tcn_mse_to_t,
    'mheme_tcn_mse_tt_t': mheme_tcn_mse_tt_t,
    'mheme_tcn_mse_e_t': mheme_tcn_mse_e_t,
    'mheme_xgb_mse_u_t': mheme_xgb_mse_u_t,
    'mheme_xgb_mse_to_t': mheme_xgb_mse_to_t,
    'mheme_xgb_mse_tt_t': mheme_xgb_mse_tt_t,
    'mheme_xgb_mse_e_t': mheme_xgb_mse_e_t,
    'tcn_hah_t': tcn_hah_t,
    'tcn_mse_t': tcn_mse_t,
    'xgb_mse_t': xgb_mse_t
}

In [ ]:
# Fit models
for name, model in models_t.items():
    print(f"Fitting model: {name}")
    train = globals()[f'train_t']
    model.fit(train[0], train[1])

In [ ]:
# Save models
for name, model in models_t.items():
    print(f"Saving model: {name}")
    model.save_model(f'../models/traffic/{name}.pkl')

### 2.4 Volatility Dataset

In [ ]:
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_tcn_hah_u_v = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_so_v = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_st_v = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_tcn_hah_e_v = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_xgb_hah_u_v  = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_so_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_st_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_xgb_hah_e_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_tcn_mse_u_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")
mheme_tcn_mse_so_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_tcn_mse_st_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")
mheme_tcn_mse_e_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_xgb_mse_u_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")   
mheme_xgb_mse_so_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_xgb_mse_st_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")    
mheme_xgb_mse_e_v = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
tcn_hah_v = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
tcn_mse_v = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
xgb_mse_v = XGBoost(horizon=HORIZON, file_path=XGB_PATH_CONFIG_LOAD)

models_v = {
    'mheme_tcn_hah_u_v': mheme_tcn_hah_u_v,
    'mheme_tcn_hah_so_v': mheme_tcn_hah_so_v,
    'mheme_tcn_hah_st_v': mheme_tcn_hah_st_v,
    'mheme_tcn_hah_e_v': mheme_tcn_hah_e_v,
    'mheme_xgb_hah_u_v': mheme_xgb_hah_u_v,
    'mheme_xgb_hah_so_v': mheme_xgb_hah_so_v,
    'mheme_xgb_hah_st_v': mheme_xgb_hah_st_v,
    'mheme_xgb_hah_e_v': mheme_xgb_hah_e_v,
    'mheme_tcn_mse_u_v': mheme_tcn_mse_u_v,
    'mheme_tcn_mse_so_v': mheme_tcn_mse_so_v,
    'mheme_tcn_mse_st_v': mheme_tcn_mse_st_v,
    'mheme_tcn_mse_e_v': mheme_tcn_mse_e_v,
    'mheme_xgb_mse_u_v': mheme_xgb_mse_u_v,
    'mheme_xgb_mse_so_v': mheme_xgb_mse_so_v,
    'mheme_xgb_mse_st_v': mheme_xgb_mse_st_v,
    'mheme_xgb_mse_e_v': mheme_xgb_mse_e_v,
    'tcn_hah_v': tcn_hah_v,
    'tcn_mse_v': tcn_mse_v,
    'xgb_mse_v': xgb_mse_t
}

In [ ]:
# Fit models
for name, model in models_v.items():
    print(f"Fitting model: {name}")
    train = globals()[f'train_v']
    model.fit(train[0], train[1])

In [ ]:
# Save models
for name, model in models_v.items():
    print(f"Saving model: {name}")
    model.save_model(f'../models/volatility/{name}.pkl')

### 2.5 Wind Dataset

In [ ]:
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_tcn_hah_u_w = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_so_w = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_tcn_hah_st_w = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_tcn_hah_e_w = UHMEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
mheme_xgb_hah_u_w  = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_so_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="horizon_weighted_huber")
mheme_xgb_hah_st_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="horizon_weighted_huber")
mheme_xgb_hah_e_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_tcn_mse_u_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")
mheme_tcn_mse_so_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_tcn_mse_st_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")
mheme_tcn_mse_e_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD)

json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
mheme_xgb_mse_u_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="soft_lin", loss_type="mse")   
mheme_xgb_mse_so_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="strong_lin", loss_type="mse")
mheme_xgb_mse_st_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="exp", loss_type="mse")    
mheme_xgb_mse_e_w = UMHEMe(window=WINDOW, horizon=HORIZON, model_class=XGBoost, config_path=XGB_PATH_CONFIG_LOAD)

json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="horizon_weighted_huber")
tcn_hah_w = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(TCN_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
tcn_mse_w = TCN(horizon=HORIZON, file_path=TCN_PATH_CONFIG_LOAD)
json_handler(XGB_PATH_CONFIG_LOAD, weights_decay="uni", loss_type="mse")
xgb_mse_w = XGBoost(horizon=HORIZON, file_path=XGB_PATH_CONFIG_LOAD)

models_w = {
    'mheme_tcn_hah_u_w': mheme_tcn_hah_u_w,
    'mheme_tcn_hah_so_w': mheme_tcn_hah_so_w,
    'mheme_tcn_hah_st_w': mheme_tcn_hah_st_w,
    'mheme_tcn_hah_e_w': mheme_tcn_hah_e_w,
    'mheme_xgb_hah_u_w': mheme_xgb_hah_u_w,
    'mheme_xgb_hah_so_w': mheme_xgb_hah_so_w,
    'mheme_xgb_hah_st_w': mheme_xgb_hah_st_w,
    'mheme_xgb_hah_e_w': mheme_xgb_hah_e_w,
    'mheme_tcn_mse_u_w': mheme_tcn_mse_u_w,
    'mheme_tcn_mse_so_w': mheme_tcn_mse_so_w,
    'mheme_tcn_mse_st_w': mheme_tcn_mse_st_w,
    'mheme_tcn_mse_e_w': mheme_tcn_mse_e_w,
    'mheme_xgb_mse_u_w': mheme_xgb_mse_u_w,
    'mheme_xgb_mse_so_w': mheme_xgb_mse_so_w,
    'mheme_xgb_mse_st_w': mheme_xgb_mse_st_w,
    'mheme_xgb_mse_e_w': mheme_xgb_mse_e_w,
    'tcn_hah_w': tcn_hah_w,
    'tcn_mse_w': tcn_mse_w,
    'xgb_mse_w': xgb_mse_t
}

In [ ]:
# Fit models
for name, model in models_w.items():
    print(f"Fitting model: {name}")
    train = globals()[f'train_w']
    model.fit(train[0], train[1])

In [ ]:
# Save models
for name, model in models_w.items():
    print(f"Saving model: {name}")
    model.save_model(f'../models/wind/{name}.pkl')

## 3. Evaluations & Plots

### 3.0 Load models

In [ ]:
# For each model in the given folder, load the model.
model_folders = {'e': '../models/electricity/', 's': '../models/solar/', 't': '../models/traffic/', 'v': '../models/volatility/', 'w': '../models/wind/'}

for key, model_folder in model_folders.items():
    loaded_models = {}
    for model_file in os.listdir(model_folder):
        if model_file.endswith('.pkl'):
            model_name = model_file[:-4]  # Remove .pkl extension
            model_path = os.path.join(model_folder, model_file)
            loaded_model = UMHEMe.load_model(model_path)
            loaded_models[model_name] = loaded_model

    # Store loaded models in a global dictionary for later use
    globals()[f'models_{key}'] = loaded_models

### 3.1 Overall Performance

In [ ]:
# Evaluate the models on the test set

# Electricity
for model in models_e:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_e']
    predictions = models_e[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

# Solar
for model in models_s:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_s']
    predictions = models_s[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

# Traffic
for model in models_t:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_t']
    predictions = models_t[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

# Volatility
for model in models_v:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_v']
    predictions = models_v[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

# Wind
for model in models_w:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_w']
    predictions = models_w[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

### 3.2 Variance of Predictors 

### 3.3 Weighting Strategy Relevance

### 3.4 Generalization